In [0]:
df = spark.sql("select * from workspace.default.tips")

In [0]:
df.show()

In [0]:
df.printSchema()

In [0]:
# Handling Categorical Features 
from pyspark.ml.feature import StringIndexer

In [0]:
indexer = StringIndexer(inputCols=["sex", "smoker", "day", "time"], outputCols=["sex_index", "smoker_index", "day_index", "time_index"])
df_reg = indexer.fit(df).transform(df)
df_reg.show()

In [0]:
df_reg.columns

In [0]:
from pyspark.ml.feature import VectorAssembler 

feature_assembler = VectorAssembler(inputCols=["tip", "size", "sex_index", "smoker_index", "day_index", "time_index"], outputCol="Independent Features")

In [0]:
output = feature_assembler.transform(df_reg)

In [0]:
output.show()

In [0]:
output.select('Independent Features').show()

In [0]:
finalized_data = output.select("Independent Features", "total_bill")

In [0]:
finalized_data.show()

In [0]:
from pyspark.ml.regression import LinearRegression 

# train-test split
train_data, test_data = finalized_data.randomSplit([0.75, 0.25])
regressor = LinearRegression(featuresCol='Independent Features', labelCol='total_bill')
regressor = regressor.fit(train_data)

In [0]:
regressor.coefficients

In [0]:
regressor.intercept

In [0]:
# Prediction
pred_result = regressor.evaluate(test_data)

In [0]:
pred_result

In [0]:
pred_result.predictions.show()

In [0]:
# Performance Metrics
pred_result.r2, pred_result.meanAbsoluteError, pred_result.meanSquaredError